# 의미 고정 N/V M5 귀속 대조실험: Dunnhumby seed 42

완료된 actual M5와 matched M4-only를 재사용하고, degree-matched joint CLV/N/V 순열과 degree loss-gate 대조군만 학습합니다. `rho=0.15`, `beta=0.25`, `lambda=0.5`를 바꾸지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '75a21ef603b2ac0defa96d7875d840d0a1a1cf68'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_semantic_nv_attribution_controls import (
    DEGREE_CONTROL_MODEL_ID,
    JOINT_SHUFFLE_MODEL_ID,
    configure_semantic_nv_attribution_controls,
    preflight_summary,
    run_semantic_nv_attribution_controls,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_semantic_nv_attribution_controls(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_semantic_nv_attribution_controls_screen_v1',
    actual_m5_result_json='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_semantic_nv_personalized_positive_single_screen_v1/m5_semantic_nv_single_426685be4486.json',
    m4_control_result_json='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_semantic_nv_m4_matched_control_screen_v1/m5_semantic_nv_m4_control_cea1a4ef5860.json',
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == [JOINT_SHUFFLE_MODEL_ID, DEGREE_CONTROL_MODEL_ID]
assert summary['fixed']['rho'] == 0.15
assert summary['fixed']['beta'] == 0.25
assert summary['fixed']['positive_weight_lambda'] == 0.5
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_semantic_nv_attribution_controls(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) M4-only·actual M5·귀속 대조군 절대지표')
show(result_df)
print('2) 각 대조군 대비 actual M5 전체 지표')
show(result_df.attrs['focused_comparison'])
print('3) CLV/N/V 귀속 판독')
print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
print('4) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))